# RAG (file_search) test

**QE Perspective:** We validate end-to-end RAG: create a vector store, upload a document, ingest it, then call the Responses API with `file_search` and assert the answer is grounded in the document. We also check **negative** (invalid vector_store_id yields an error) and **edge** (missing config, no vector_io provider fail fast). This ensures the RAG pipeline and API contract are stable.

- **Positive:** Vector store + file upload + file_search → response contains expected fact (e.g. "1980").
- **Negative:** Request with non-existent vector_store_id → API returns error.
- **Edge:** Assert base_url/model set; assert at least one vector_io provider before running.

Aligned with [llama-stack-demos simple_rag](https://github.com/opendatahub-io/llama-stack-demos/blob/main/demos/03_rag/01_simple_rag.py). Config: `BASE_URL`, `MODEL` (optional: `EMBEDDING_MODEL`, `EMBEDDING_DIMENSION`). Run via pytest or interactively.


## Setup

Load config from env; optionally import `response_text` from `scripts.helpers` for consistent response parsing. Create client and ensure vector_io provider exists.


In [ ]:
import os
from scripts.helpers import response_text


base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))

_skip_reason = None

# Edge: fail fast if config missing
assert base_url, "BASE_URL must be set (e.g. http://localhost:8321)"
assert model, (
    "MODEL env var is required. Set it before running: "
    "export MODEL=your-model-id"
)
if not embedding_model:
    _skip_reason = "EMBEDDING_MODEL env var not set — RAG test requires an embedding model"
    print(f"SKIPPED: {_skip_reason}")

In [ ]:
if not _skip_reason:
    from llama_stack_client import LlamaStackClient

    client = LlamaStackClient(base_url=base_url)

    models = client.models.list()
    model_ids = [m.id for m in models]
    assert model in model_ids, f"Model {model} not found. Available: {model_ids}"
    assert embedding_model in model_ids, f"Embedding model {embedding_model} not found. Available: {model_ids}"
    print(f"Using model: {model}")
    print(f"Using embedding model: {embedding_model}")

In [ ]:
if not _skip_reason:
    # Edge: no vector_io provider → cannot run RAG
    vector_providers = [p for p in client.providers.list() if p.api == "vector_io"]
    if not vector_providers:
        _skip_reason = "No vector_io provider available — RAG test cannot run"
        print(f"SKIPPED: {_skip_reason}")
    else:
        selected_vector_provider = vector_providers[0]

In [ ]:
if not _skip_reason:
    # Positive: create vector store, upload doc, file_search, assert answer
    from io import BytesIO
    from uuid import uuid4

    doc_text = """Bering Land Bridge National Preserve. Proclaimed a national monument Dec. 1, 1978; established as a national preserve Dec. 2, 1980.
    Denali National Park. Established as Mt. McKinley National Park Feb. 26, 1917. Designated Denali National Park and Preserve Dec. 2, 1980."""
    question = "When was Bering Land Bridge established as a national preserve?"

    vector_store = None
    uploaded_file = None
    try:
        vector_store = client.vector_stores.create(
            name=f"rag_test_{uuid4().hex[:8]}",
            extra_body={
                "provider_id": selected_vector_provider.provider_id,
                "embedding_model": embedding_model,
                "embedding_dimension": embedding_dimension,
            },
        )
        file_buffer = BytesIO(doc_text.encode("utf-8"))
        file_buffer.name = "rag_doc.txt"
        uploaded_file = client.files.create(file=file_buffer, purpose="assistants")
        client.vector_stores.files.create(
            vector_store_id=vector_store.id,
            file_id=uploaded_file.id,
            chunking_strategy={
                "type": "static",
                "static": {"max_chunk_size_tokens": 256, "chunk_overlap_tokens": 32},
            },
        )
        response = client.responses.create(
            model=model,
            instructions="Use file_search to answer the question using the provided documents.",
            input=[{"role": "user", "content": question}],
            tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
            tool_choice={"type": "file_search"},
            stream=False,
        )
        assert (
            response.status == "completed"
        ), f"Expected status completed, got {response.status}"
        text = response_text(response)
        if not text:
            text = getattr(response, "output_text", "") or ""
        assert "1980" in text, f"Expected '1980' in answer from doc, got: {text[:300]}"
    finally:
        if vector_store:
            try:
                client.vector_stores.delete(vector_store_id=vector_store.id)
            except Exception:
                pass
        if uploaded_file:
            try:
                client.files.delete(file_id=uploaded_file.id)
            except Exception:
                pass

In [ ]:
if not _skip_reason:
    # Negative: file_search with invalid vector_store_id should fail
    raised = False
    try:
        client.responses.create(
            model=model,
            input=[{"role": "user", "content": "What is 2+2?"}],
            tools=[{"type": "file_search", "vector_store_ids": ["vs_nonexistent_invalid"]}],
            tool_choice={"type": "file_search"},
            stream=False,
        )
    except Exception as e:
        raised = True
    assert raised, "Expected an error when using invalid vector_store_id"

In [ ]:
# RAG test done: positive (file_search with real store), negative (invalid store id), edge (config + no provider)